# 01 — Análise Exploratória da Série Temporal de Vendas

Este notebook é a primeira etapa do laboratório de forecasting da AdventureWorks.

## Objetivo

Antes de prever o futuro, precisamos entender o comportamento histórico da série.

Ao final deste notebook, devemos conseguir responder:

- Qual é o período disponível?
- Existem meses ausentes?
- A receita apresenta tendência?
- Há indícios de sazonalidade?
- Existem picos ou quedas fora do padrão?
- Como se comportam receita, pedidos e unidades vendidas ao longo do tempo?

> Neste notebook **não vamos criar nenhum modelo de forecast**.

## 1. Bibliotecas

Vamos começar com poucas dependências:

- `pandas`: manipulação dos dados;
- `sqlalchemy`: conexão com PostgreSQL;
- `matplotlib`: visualização.

A ideia é manter o laboratório simples antes de introduzir bibliotecas específicas de séries temporais.

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ModuleNotFoundError: No module named 'pandas'

## 2. Conexão com o Data Warehouse

O ideal é não colocar usuário e senha diretamente no notebook.

Use variáveis de ambiente. Exemplo:

```bash
DB_USER=analytics
DB_PASSWORD=analytics
DB_HOST=warehouse
DB_PORT=5432
DB_NAME=analytics
```

Se o notebook estiver rodando fora do Docker, o `DB_HOST` e a porta podem ser diferentes.

Ajuste conforme a configuração atual do projeto.

In [ ]:
DB_USER = "analytics_user"
DB_PASSWORD = "analytics_dev"
DB_HOST = "localhost"
DB_PORT = "5434"
DB_NAME = "analytics"

DATABASE_URL = (
    f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

## 3. Carregamento da série mensal

Nossa fonte oficial para este laboratório será:

`analytics.fct_sales_monthly`

Não vamos reconstruir as regras de negócio no notebook.

In [ ]:
query = '''
select
    *
from analytics.fct_sales_monthly
order by sales_month
'''

df = pd.read_sql(query, engine)

df.head()

## 4. Primeira inspeção

Antes de qualquer gráfico ou estatística, precisamos entender a estrutura dos dados.

In [ ]:
print(f"Linhas: {len(df)}")
print("\nColunas:")
print(df.columns.tolist())

print("\nTipos:")
display(df.dtypes)

print("\nPrimeiras linhas:")
display(df.head())

print("\nÚltimas linhas:")
display(df.tail())

### Perguntas para responder

- Cada linha realmente representa um único mês?
- `sales_month` foi interpretado como data?
- As métricas parecem coerentes?

## 5. Preparação da coluna temporal

In [ ]:
df["sales_month"] = pd.to_datetime(df["sales_month"])

df = df.sort_values("sales_month").reset_index(drop=True)

print("Primeiro mês:", df["sales_month"].min())
print("Último mês:", df["sales_month"].max())
print("Quantidade de meses:", len(df))

## 6. Validação da continuidade mensal

Forecasting funciona melhor quando sabemos exatamente qual é a frequência da série.

Aqui vamos verificar se existe algum mês ausente entre o início e o fim do histórico.

In [ ]:
expected_months = pd.date_range(
    start=df["sales_month"].min(),
    end=df["sales_month"].max(),
    freq="MS"
)

missing_months = expected_months.difference(df["sales_month"])

if len(missing_months) == 0:
    print("✅ A série mensal é contínua.")
else:
    print("⚠️ Meses ausentes:")
    display(pd.DataFrame({"missing_month": missing_months}))

## 7. Estatísticas descritivas

Essas estatísticas não explicam a série temporal, mas ajudam a identificar escala, dispersão e possíveis valores extremos.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

display(df[numeric_columns].describe().T)

## 8. Receita mensal ao longo do tempo

Este é o primeiro gráfico realmente importante do laboratório.

Observe principalmente:

- direção geral;
- oscilações;
- picos;
- quedas;
- padrões que parecem se repetir.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df["sales_month"], df["net_sales"], marker="o")
plt.title("AdventureWorks — Receita mensal")
plt.xlabel("Mês")
plt.ylabel("Receita líquida")
plt.grid(alpha=0.3)
plt.show()

### Anotações

Depois de executar o gráfico, registre abaixo o que você observa.

Exemplos de perguntas:

- A receita parece crescer ao longo do tempo?
- Existe algum período claramente mais forte?
- Há algum mês muito diferente dos demais?
- O comportamento parece aleatório ou existe algum padrão?

**Minhas observações:**

- 
- 
-

## 9. Médias móveis

A média móvel reduz parte do ruído mensal e facilita a visualização da tendência.

Vamos comparar:

- série original;
- média móvel de 3 meses;
- média móvel de 6 meses.

In [ ]:
df["net_sales_ma_3"] = df["net_sales"].rolling(window=3).mean()
df["net_sales_ma_6"] = df["net_sales"].rolling(window=6).mean()

plt.figure(figsize=(14, 6))
plt.plot(df["sales_month"], df["net_sales"], label="Receita mensal", alpha=0.6)
plt.plot(df["sales_month"], df["net_sales_ma_3"], label="Média móvel 3 meses")
plt.plot(df["sales_month"], df["net_sales_ma_6"], label="Média móvel 6 meses")

plt.title("Receita mensal e médias móveis")
plt.xlabel("Mês")
plt.ylabel("Receita líquida")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Conceito: tendência

Se as médias móveis sobem ou descem de maneira relativamente consistente, isso pode indicar uma **tendência**.

Não estamos provando estatisticamente nada ainda. Estamos apenas procurando evidências visuais.

## 10. Comparação com o mesmo mês do ano anterior

Uma das primeiras formas de investigar sazonalidade é comparar cada mês com o mesmo mês do ano anterior.

In [ ]:
df["net_sales_yoy_pct"] = df["net_sales"].pct_change(periods=12) * 100

display(
    df[
        ["sales_month", "net_sales", "net_sales_yoy_pct"]
    ].tail(18)
)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["sales_month"], df["net_sales_yoy_pct"], marker="o")
plt.axhline(0, linewidth=1)
plt.title("Variação da receita vs. mesmo mês do ano anterior")
plt.xlabel("Mês")
plt.ylabel("Variação YoY (%)")
plt.grid(alpha=0.3)
plt.show()

## 11. Receita por mês do ano

Agora vamos agrupar todos os janeiros, fevereiros, marços etc.

Isso ajuda a investigar se determinados meses tendem a ser mais fortes ou mais fracos.

In [ ]:
df["month_number"] = df["sales_month"].dt.month
df["month_name"] = df["sales_month"].dt.strftime("%b")

monthly_pattern = (
    df.groupby(["month_number", "month_name"], as_index=False)
      .agg(
          avg_net_sales=("net_sales", "mean"),
          median_net_sales=("net_sales", "median"),
          observations=("net_sales", "size")
      )
      .sort_values("month_number")
)

display(monthly_pattern)

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(monthly_pattern["month_name"], monthly_pattern["avg_net_sales"])
plt.title("Receita média por mês do ano")
plt.xlabel("Mês")
plt.ylabel("Receita média")
plt.show()

### Atenção

Temos poucos anos de histórico.

Isso significa que devemos ser cuidadosos antes de afirmar que existe sazonalidade apenas porque alguns meses apresentaram valores maiores.

Neste estágio, estamos procurando **indícios**, não conclusões definitivas.

## 12. Pedidos e unidades vendidas

Receita pode variar por diferentes motivos.

Por isso vamos observar também:

- quantidade de pedidos;
- quantidade de unidades vendidas.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["sales_month"], df["orders_count"], marker="o")
ax.set_title("Pedidos por mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Pedidos")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["sales_month"], df["units_sold"], marker="o")
ax.set_title("Unidades vendidas por mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Unidades")
ax.grid(alpha=0.3)
plt.show()

## 13. Ticket médio

Se a tabela possui `average_order_value`, podemos investigar se o crescimento da receita vem de:

- mais pedidos;
- pedidos maiores;
- ou ambos.

In [ ]:
if "average_order_value" in df.columns:
    plt.figure(figsize=(14, 5))
    plt.plot(df["sales_month"], df["average_order_value"], marker="o")
    plt.title("Ticket médio mensal")
    plt.xlabel("Mês")
    plt.ylabel("Ticket médio")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Coluna average_order_value não encontrada.")

## 14. Identificação simples de meses extremos

Ainda não estamos fazendo detecção formal de outliers.

Vamos apenas observar os meses de maior e menor receita.

In [ ]:
top_months = df.nlargest(5, "net_sales")[
    ["sales_month", "net_sales", "orders_count", "units_sold"]
]

bottom_months = df.nsmallest(5, "net_sales")[
    ["sales_month", "net_sales", "orders_count", "units_sold"]
]

print("Maiores meses:")
display(top_months)

print("Menores meses:")
display(bottom_months)

## 15. Resumo do diagnóstico

Complete esta seção depois de analisar os resultados.

### Período disponível

- Primeiro mês:
- Último mês:
- Número de observações:

### Tendência

- Há evidência visual de tendência?
- Crescimento, estabilidade ou queda?

### Sazonalidade

- Existem meses que parecem consistentemente mais fortes ou fracos?
- O histórico disponível é suficiente para ter confiança nessa conclusão?

### Valores extremos

- Quais meses chamaram atenção?
- Existe alguma possível explicação de negócio?

### Próximo passo

Depois deste notebook, o próximo laboratório será:

**02_baselines.ipynb**

Nele vamos aprender a criar previsões extremamente simples:

1. Naive;
2. Seasonal Naive;
3. Média móvel.

Esses modelos serão nossa referência para avaliar modelos mais sofisticados posteriormente.